
# **English → *Konkani* Health Domain Machine Translation System**
### Fine-tuned IndicTrans2 | All 7 Research Papers Applied
---
**Before running anything:**
1. Go to **Runtime → Change Runtime Type → T4 GPU**
2. Run cells **one by one in order**
3. Your model saves automatically to **Google Drive → Radhika_model**
4. If Colab disconnects, run Cell 1, 2, 3 then jump to Cell 14 — it resumes automatically

---
**Research Papers Used:**
- Paper 1 (Dewa 2018) — Healthcare MT domain importance
- Paper 2 (Ugas 2025) — MT vs Human evaluation metrics
- Paper 3 (Patil & Davies 2014) — Back-translation quality check
- Paper 4 (Neves 2023) — Multi-metric evaluation
- Paper 5 (Gala 2023) — IndicTrans2 model
- Paper 6 (Kangle 2016) — Rule-based Konkani MT baseline
- Paper 7 (Patil & Naik 2018) — Hybrid Konkani MT baseline
- Vidyaapati Workshop (Goa University) — NER + MWE + POS annotation
- Paper on verb chunking (TAM system for Indian languages)
---

## 📁 CELL 1 — Mount Google Drive & Create Radhika_model Folder

**What this does:** Connects your Google Drive to this Colab session and creates the `Radhika_model` folder structure where all your checkpoints, processed data, tokenizer, and final model will be saved permanently. Even if Colab disconnects, everything is safe in your Drive.

When you run this cell a popup will appear asking you to allow Drive access — click Allow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR = '/content/drive/MyDrive/Radhika_model'

os.makedirs(f'{DRIVE_DIR}/checkpoints',      exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/final_model',      exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results',          exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data',             exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/tokenizer',        exist_ok=True)

print('✅ Google Drive mounted!')
print(f'✅ Radhika_model folder ready at: {DRIVE_DIR}')
print('\nFolders created:')
for f in os.listdir(DRIVE_DIR):
    print(f'  📁 {f}')

Mounted at /content/drive
✅ Google Drive mounted!
✅ Radhika_model folder ready at: /content/drive/MyDrive/Radhika_model

Folders created:
  📁 results
  📁 data
  📁 tokenizer
  📁 checkpoints
  📁 final_model


## 📦 CELL 2 — Install All Required Libraries

**What this does:** Installs every Python library the system needs.
- `transformers` — loads and fine-tunes IndicTrans2 (Paper 5)
- `sentencepiece` — trains custom BPE tokenizer on your data
- `sacrebleu` — calculates BLEU and chrF scores (Papers 2, 4)
- `stanza` — POS tagging for verb chunking (verb chunking paper)
- `datasets` — formats data for HuggingFace training
- `accelerate` — GPU training optimization
- `indic-nlp-library` — Devanagari/Konkani text tools

**⏱ Time:** 3–5 minutes. Wait for the green tick before moving on.

In [ ]:
# PERMANENT FIX — Run this as the very first cell
# The issue is transformers 4.41.0 still has EncoderDecoderCache
# conflict on Python 3.12. We need 4.44.0 or higher.

!pip install transformers==4.44.2 -q
!pip install accelerate==0.33.0 -q
!pip install sentencepiece==0.1.99 -q
!pip install sacrebleu==2.3.1 -q
!pip install stanza==1.7.0 -q
!pip install datasets==2.17.0 -q
!pip install evaluate -q
!pip install indic-nlp-library -q
!pip install scikit-learn -q

print("✅ All libraries installed with correct versions!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>

In [ ]:
# FIX — NumPy binary incompatibility
# Must fix numpy FIRST before anything else

!pip uninstall numpy -y -q
!pip install numpy==1.26.4 -q

!pip uninstall transformers -y -q
!pip install transformers==4.44.2 -q
!pip install accelerate==0.33.0 -q
!pip install sentencepiece==0.1.99 -q
!pip install sacrebleu==2.3.1 -q
!pip install stanza==1.7.0 -q
!pip install datasets==2.17.0 -q
!pip install evaluate -q
!pip install indic-nlp-library -q
!pip install scikit-learn -q

print("✅ Done! Now go to Runtime → Restart session")
print("⚠️  You MUST restart runtime before running the next cell")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires nu

In [ ]:
# Run this AFTER restarting runtime
import numpy as np
import torch
import transformers

print(f"NumPy version        : {np.__version__}")
print(f"Transformers version : {transformers.__version__}")
print(f"PyTorch version      : {torch.__version__}")
print(f"GPU available        : {torch.cuda.is_available()}")

from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    EarlyStoppingCallback
)

print("\n✅ NumPy fixed!")
print("✅ All imports working!")
print("✅ Ready — continue from Cell 3 onwards")

NumPy version        : 1.26.4
Transformers version : 4.44.2
PyTorch version      : 2.10.0+cu128
GPU available        : True

✅ NumPy fixed!
✅ All imports working!
✅ Ready — continue from Cell 3 onwards


## 🔧 CELL 3 — Import All Libraries & Check GPU

**What this does:** Loads every library into memory and checks your GPU. If it prints 'NO GPU FOUND' you must go to Runtime → Change Runtime Type → T4 GPU and restart before continuing. Training without GPU will take days instead of hours.

In [ ]:
import os, re, json, unicodedata, warnings, shutil, glob
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import torch
import stanza
import sentencepiece as spm

from datasets import Dataset, load_from_disk
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from sacrebleu.metrics import BLEU, CHRF
import evaluate as hf_evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {device}')
if torch.cuda.is_available():
    print(f'GPU Name   : {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Memory : {mem:.1f} GB')
else:
    print('⚠️  NO GPU — Go to Runtime → Change Runtime Type → T4 GPU!')

DRIVE_DIR = '/content/drive/MyDrive/Radhika_model'
print(f'\n✅ All imports done!')
print(f'✅ Drive folder: {DRIVE_DIR}')

Device     : cuda
GPU Name   : Tesla T4
GPU Memory : 15.6 GB

✅ All imports done!
✅ Drive folder: /content/drive/MyDrive/Radhika_model


## 📤 CELL 4 — Upload Your Two CSV Files

**What this does:** Opens a file upload dialog. Click 'Choose Files' and select BOTH your CSV files at the same time — the English one and the Konkani one.

After uploading, this cell will show you the first 3 lines of each file so you can confirm which file is English and which is Konkani. You will use this information in Cell 5.

In [ ]:
from google.colab import files
import pandas as pd

print('📤 Upload your TWO xlsx files now...')
print('   → Your English xlsx')
print('   → Your Konkani xlsx')
print()

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

print(f'\n✅ Files uploaded: {uploaded_files}')
print('\n--- FILE PREVIEW ---')
for fname in uploaded_files:
    # ✅ FIX: Use read_excel for .xlsx files, not read_csv
    temp = pd.read_excel(fname, header=None, names=['text'])
    print(f'\n📄 {fname}  ({len(temp):,} rows)')
    print(f'  Line 1: {temp["text"].iloc[0]}')
    print(f'  Line 2: {temp["text"].iloc[1]}')
    print(f'  Line 3: {temp["text"].iloc[2]}')

📤 Upload your TWO xlsx files now...
   → Your English xlsx
   → Your Konkani xlsx



Saving ENGLISH_DATA.xlsx to ENGLISH_DATA.xlsx
Saving KONKANI_DATA.xlsx to KONKANI_DATA.xlsx

✅ Files uploaded: ['ENGLISH_DATA.xlsx', 'KONKANI_DATA.xlsx']

--- FILE PREVIEW ---

📄 ENGLISH_DATA.xlsx  (24,923 rows)
  Line 1: Fresh breath and shining teeth enhance your personality .
  Line 2: Your self-confidence also increases with teeth .
  Line 3: Bacteria stay between our gums and teeth .

📄 KONKANI_DATA.xlsx  (24,923 rows)
  Line 1: ताजो स्वास आनी चकचकीत दांत तुमचें व्यक्तीमत्व परजळायतात .

  Line 2: दांत आशिल्ल्यान तुमचो आत्मविश्वासय वाडटा .

  Line 3: आमच्या हड्ड्यां आनी दांतां मदीं बॅक्टेरिया आसतात .



## 🔗 CELL 5 — Pair Files Into Parallel Corpus

**What this does:** Pairs your English and Konkani files into one parallel dataframe. Line 1 of the English file pairs with Line 1 of the Konkani file, all the way to line 25,000.

**⚠️ Action needed:** Look at the preview from Cell 4. If the first file shown is Konkani (not English), swap the index numbers — change `uploaded_files[0]` to `uploaded_files[1]` and vice versa on the two lines marked below.

In [ ]:
# ✏️ Check Cell 4 output. If files are in wrong order, swap [0] and [1] below
ENGLISH_FILE = uploaded_files[0]
KONKANI_FILE = uploaded_files[1]

# ✅ FIX: Use read_excel instead of read_csv
en_df  = pd.read_excel(ENGLISH_FILE, header=None, names=['text'])
kok_df = pd.read_excel(KONKANI_FILE, header=None, names=['text'])

df = pd.DataFrame({
    'english': en_df['text'].astype(str).str.strip(),
    'konkani': kok_df['text'].astype(str).str.strip()
})

print(f'✅ Parallel corpus created!')
print(f'   English file : {ENGLISH_FILE}')
print(f'   Konkani file : {KONKANI_FILE}')
print(f'   Total pairs  : {len(df):,}')
print(f'\nSample pairs:')
for i in range(3):
    print(f'\n  [{i+1}] EN  : {df["english"].iloc[i]}')
    print(f'       KOK : {df["konkani"].iloc[i]}')

✅ Parallel corpus created!
   English file : ENGLISH_DATA.xlsx
   Konkani file : KONKANI_DATA.xlsx
   Total pairs  : 24,923

Sample pairs:

  [1] EN  : Fresh breath and shining teeth enhance your personality .
       KOK : ताजो स्वास आनी चकचकीत दांत तुमचें व्यक्तीमत्व परजळायतात .

  [2] EN  : Your self-confidence also increases with teeth .
       KOK : दांत आशिल्ल्यान तुमचो आत्मविश्वासय वाडटा .

  [3] EN  : Bacteria stay between our gums and teeth .
       KOK : आमच्या हड्ड्यां आनी दांतां मदीं बॅक्टेरिया आसतात .


## 🧹 CELL 6 — Clean Your Data

**What this does (Paper 1 — Dewa 2018 + Verb Chunking Paper preprocessing approach):**

**English cleaning:** Converts to lowercase, expands contractions like 'don't' → 'do not', removes special characters and extra whitespace. Paper 1 showed that text normalization before translation significantly improves quality.

**Konkani cleaning:** Applies Unicode NFC normalization — this is critical for Devanagari script because the same character can be stored in multiple ways internally that look identical on screen but are different to the model. Normalization ensures consistency.

**Filtering:** Removes empty rows, duplicates, and sentences that are too short (under 3 words) or too long (over 60 words) because extremely long sentences are harder to align correctly.

In [ ]:
import re
import unicodedata  # also needed for clean_konkani in this same cell

def clean_english(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    contractions = {
        "n't":" not","'s":" is","'re":" are","'ve":" have",
        "'ll":" will","'d":" would","can't":"cannot",
        "won't":"will not","don't":"do not","doesn't":"does not",
        "didn't":"did not","shouldn't":"should not",
        "isn't":"is not","aren't":"are not","hasn't":"has not",
        "couldn't":"could not","wouldn't":"would not",
    }
    for c, e in contractions.items():
        text = text.replace(c, e)
    text = re.sub(r'[^a-z0-9\s\.,!?\-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ... rest of Cell 6 unchanged

def clean_konkani(text):
    if not isinstance(text, str): return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[^\u0900-\u097F\s\.,!?।\-0-9]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

before = len(df)
df['english'] = df['english'].apply(clean_english)
df['konkani'] = df['konkani'].apply(clean_konkani)

df = df[df['english'].str.len() > 5]
df = df[df['konkani'].str.len()  > 5]
df = df.drop_duplicates(subset=['english'])
df = df.drop_duplicates(subset=['konkani'])

df['en_len']  = df['english'].apply(lambda x: len(x.split()))
df['kok_len'] = df['konkani'].apply(lambda x: len(x.split()))
df = df[(df['en_len']  >= 3) & (df['en_len']  <= 60)]
df = df[(df['kok_len'] >= 3) & (df['kok_len'] <= 70)]
df = df.drop(columns=['en_len','kok_len']).reset_index(drop=True)

print(f'Before cleaning : {before:,}')
print(f'After cleaning  : {len(df):,}')
print(f'Removed         : {before - len(df):,} noisy pairs')
print(f'\nClean example:')
print(f'  EN  : {df["english"].iloc[0]}')
print(f'  KOK : {df["konkani"].iloc[0]}')
print('\n✅ Data cleaning complete!')

Before cleaning : 24,923
After cleaning  : 24,743
Removed         : 180 noisy pairs

Clean example:
  EN  : fresh breath and shining teeth enhance your personality .
  KOK : ताजो स्वास आनी चकचकीत दांत तुमचें व्यक्तीमत्व परजळायतात .

✅ Data cleaning complete!


## ✂️ CELL 7 — Train / Dev / Test Split & Save to Drive

**What this does:** Divides your data into three parts and saves them permanently to Drive.

- **Training set (80% ≈ 20,000 pairs)** — what the model learns from during fine-tuning
- **Dev set (10% ≈ 2,500 pairs)** — used to measure BLEU score after each epoch during training so we know when to stop
- **Test set (10% ≈ 2,500 pairs)** — kept completely hidden, only used at the very end for your final official BLEU score

This follows the benchmark evaluation framework from Paper 4 (Neves 2023).

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/Radhika_model'

# Re-mount Drive and recreate folders if needed
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
dev_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df = train_df.reset_index(drop=True)
dev_df   = dev_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df.to_csv(f'{DRIVE_DIR}/data/train.csv', index=False)
dev_df.to_csv(  f'{DRIVE_DIR}/data/dev.csv',   index=False)
test_df.to_csv( f'{DRIVE_DIR}/data/test.csv',  index=False)

print('✅ Data split complete!')
print(f'  Training set : {len(train_df):,} pairs  (80%)')
print(f'  Dev set      : {len(dev_df):,} pairs  (10%)')
print(f'  Test set     : {len(test_df):,} pairs  (10%)')
print(f'\n✅ All splits saved to Drive!')

Mounted at /content/drive
✅ Data split complete!
  Training set : 19,794 pairs  (80%)
  Dev set      : 2,474 pairs  (10%)
  Test set     : 2,475 pairs  (10%)

✅ All splits saved to Drive!


## 🏷️ CELL 8 — NER Protection



Health terms like 'omega-3', 'vitamin D', 'oats', 'quinoa' have no standard Konkani equivalent. If the model tries to translate them it produces wrong or made-up words. The solution: replace them with a placeholder like `__ENT5__` before translation, then restore the original after. The model translates everything correctly around the protected term.

Numbers with units like '200 calories' or '30 minutes' are also protected so they stay accurate in the output.

In [ ]:
# — NER Protection
DO_NOT_TRANSLATE = [
    'chia seeds','flaxseed','almond flour','coconut flour',
    'almond milk','coconut oil','olive oil','oat flour',
    'quinoa','stevia','oats','millet','sorghum',
    'baking powder','baking soda','vanilla extract',
    'vitamin a','vitamin b','vitamin c','vitamin d','vitamin e',
    'omega-3','omega-6','omega 3','omega 6',
    'zinc','magnesium','calcium','potassium',
    'antioxidant','antioxidants','probiotics','prebiotics',
    'glycemic index','insulin','triglycerides',
    'hdl','ldl','bmi','ibs','dha','epa',
]

def protect_entities(text):
    entity_map = {}
    masked = text
    for i, ent in enumerate(sorted(DO_NOT_TRANSLATE, key=len, reverse=True)):
        pattern = r'\b' + re.escape(ent) + r'\b'
        if re.search(pattern, masked, re.IGNORECASE):
            ph = f'__ENT{i}__'
            masked = re.sub(pattern, ph, masked, flags=re.IGNORECASE)
            entity_map[ph] = ent
    num_pat = r'\b(\d+\.?\d*)\s*(g|kg|ml|cal|kcal|mg|calories?|grams?|minutes?|hours?|days?)\b'
    for j, m in enumerate(re.finditer(num_pat, masked, re.IGNORECASE)):
        ph = f'__NUM{j}__'
        entity_map[ph] = m.group()
        masked = masked.replace(m.group(), ph, 1)
    return masked, entity_map

def restore_entities(text, entity_map):
    for ph, orig in entity_map.items():
        text = text.replace(ph, orig)
    return text

# Test
s = 'take 500mg vitamin c with chia seeds and oats daily'
masked, mapping = protect_entities(s)
print('NER Protection Test (Vidyaapati Workshop):')
print(f'  Original : {s}')
print(f'  Masked   : {masked}')
print(f'  Mapping  : {mapping}')
print('\n✅ NER protection ready!')

NER Protection Test (Vidyaapati Workshop):
  Original : take 500mg vitamin c with chia seeds and oats daily
  Masked   : take __NUM0__ __ENT18__ with __ENT11__ and __ENT34__ daily
  Mapping  : {'__ENT11__': 'chia seeds', '__ENT18__': 'vitamin c', '__ENT34__': 'oats', '__NUM0__': '500mg'}

✅ NER protection ready!


## 🔤 CELL 9 — MWE Detection

For example 'blood pressure' is not 'blood' plus 'pressure' independently — it is one medical concept. If the model sees them as two separate tokens it may translate them separately and produce a wrong result. By joining them as 'blood_pressure' the model learns to treat the whole phrase as one unit and translates it correctly every time.

This directly improves BLEU score for health domain sentences because medical compound terms are very frequent in your training data.

In [ ]:
# Vp — MWE Joining
ENGLISH_MWE = {
    'blood pressure':'blood_pressure',
    'blood sugar':'blood_sugar',
    'blood glucose':'blood_glucose',
    'heart rate':'heart_rate',
    'heart disease':'heart_disease',
    'body weight':'body_weight',
    'body mass index':'body_mass_index',
    'immune system':'immune_system',
    'digestive system':'digestive_system',
    'healthy doughnut':'healthy_doughnut',
    'whole wheat':'whole_wheat',
    'whole grain':'whole_grain',
    'brown sugar':'brown_sugar',
    'low calorie':'low_calorie',
    'high fiber':'high_fiber',
    'low fat':'low_fat',
    'sugar free':'sugar_free',
    'gluten free':'gluten_free',
    'dairy free':'dairy_free',
    'nutritional value':'nutritional_value',
    'dietary fiber':'dietary_fiber',
    'saturated fat':'saturated_fat',
    'trans fat':'trans_fat',
    'physical activity':'physical_activity',
    'weight loss':'weight_loss',
    'weight gain':'weight_gain',
    'calorie intake':'calorie_intake',
    'balanced diet':'balanced_diet',
    'healthy eating':'healthy_eating',
    'meal plan':'meal_plan',
    'portion control':'portion_control',
    'regular exercise':'regular_exercise',
    'baking powder':'baking_powder',
    'deep fry':'deep_fry',
    'olive oil':'olive_oil',
    'coconut oil':'coconut_oil',
}

def apply_mwe(text):
    for mwe, joined in sorted(ENGLISH_MWE.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r'\b' + re.escape(mwe) + r'\b'
        text = re.sub(pattern, joined, text, flags=re.IGNORECASE)
    return text

s = 'a healthy doughnut with low calorie ingredients helps manage blood sugar'
print('MWE Test (Vidyaapati Workshop):')
print(f'  Before : {s}')
print(f'  After  : {apply_mwe(s)}')
print('\n✅ MWE detection ready!')

MWE Test (Vidyaapati Workshop):
  Before : a healthy doughnut with low calorie ingredients helps manage blood sugar
  After  : a healthy_doughnut with low_calorie ingredients helps manage blood_sugar

✅ MWE detection ready!


## 🔍 CELL 10 — POS Tagging & Verb Chunking (Verb Chunking Paper + Papers 6 & 7)

**What this does:** This is the core method from the verb chunking paper — co-joining verbal constructs.

The paper found that when an MT system sees 'should be taking' as three separate words it often mistranslates the verb because it does not understand they form one unified meaning — obligation + continuous action together. The solution is POS tagging each word (labeling it as Noun, Verb, Auxiliary etc.) and then joining consecutive auxiliaries and main verbs with underscore. So 'should be taking' becomes 'should_be_taking' — one single token with one unified meaning.

The Konkani TAM (Tense-Aspect-Modality) patterns come from Papers 6 and 7 — the Konkani-specific papers by Kangle (2016) and Patil & Naik (2018) which documented how Konkani verb groups work grammatically.

**⏱ Time:** About 2 minutes to download Stanza model.

In [ ]:
!pip install stanza==1.7.0 -q

import stanza
import re
import torch
import numpy as np

# ── FIX: Allow numpy globals that PyTorch 2.6 blocks by default ──
import torch.serialization
import numpy.core.multiarray
torch.serialization.add_safe_globals([
    numpy.core.multiarray._reconstruct,
    numpy.ndarray,
    numpy.dtype,
    np.core.multiarray._reconstruct,
])

# ── FIX: Patch torch.load so Stanza can load its models ──
original_torch_load = torch.load
def patched_torch_load(f, *args, **kwargs):
    kwargs['weights_only'] = False
    return original_torch_load(f, *args, **kwargs)
torch.load = patched_torch_load

# ── Now download and load Stanza safely ──
print('Downloading Stanza English POS model...')
stanza.download('en', verbose=False)
nlp_en = stanza.Pipeline('en', processors='tokenize,pos', verbose=False)
print('✅ Stanza loaded!')

# ── Restore original torch.load after Stanza is done ──
torch.load = original_torch_load

# ── Verb Chunking Paper + Papers 6 & 7 (Konkani-specific grammar) ──
MODALS = {'can','could','should','would','must','may','might','will','shall','ought'}

def chunk_english_verbs(text):
    """Verb Chunking Paper — join multi-word verb phrases"""
    try:
        doc = nlp_en(text)
        result = []
        for sent in doc.sentences:
            words = sent.words
            i = 0
            while i < len(words):
                w = words[i]
                if (i+2 < len(words)
                        and w.upos=='AUX'
                        and words[i+1].upos=='AUX'
                        and words[i+2].upos=='VERB'):
                    result.append(f'{w.text}_{words[i+1].text}_{words[i+2].text}')
                    i += 3
                elif (i+1 < len(words)
                        and w.upos=='AUX'
                        and words[i+1].upos=='VERB'):
                    result.append(f'{w.text}_{words[i+1].text}')
                    i += 2
                elif (i+1 < len(words)
                        and w.text.lower() in MODALS
                        and words[i+1].upos=='VERB'):
                    result.append(f'{w.text}_{words[i+1].text}')
                    i += 2
                else:
                    result.append(w.text)
                    i += 1
        return ' '.join(result)
    except:
        return text

# ── Konkani TAM patterns — Papers 6 & 7 (Kangle 2016, Patil & Naik 2018) ──
KOK_TAM = [
    (r'([\u0900-\u097F]+ता)\s+(आसा)',    r'\1_\2'),
    (r'([\u0900-\u097F]+ता)\s+(आसलो)',   r'\1_\2'),
    (r'([\u0900-\u097F]+लां)\s+(आसा)',   r'\1_\2'),
    (r'([\u0900-\u097F]+ून)\s+(घेता)',   r'\1_\2'),
    (r'([\u0900-\u097F]+ून)\s+(दिता)',   r'\1_\2'),
    (r'([\u0900-\u097F]+)\s+(जाय)',      r'\1_\2'),
    (r'([\u0900-\u097F]+)\s+(ना)',       r'\1_\2'),
    (r'([\u0900-\u097F]+तलो)\s+(आसा)',  r'\1_\2'),
]

def chunk_konkani_verbs(text):
    for pat, rep in KOK_TAM:
        text = re.sub(pat, rep, text)
    return text

print('\nVerb Chunking Tests:')
print(f'  EN before : patient should be taking medicine daily')
print(f'  EN after  : {chunk_english_verbs("patient should be taking medicine daily")}')
print(f'\n  KOK before: ताणें दवो घेता आसा')
print(f'  KOK after : {chunk_konkani_verbs("ताणें दवो घेता आसा")}')
print('\n✅ Verb chunking ready!')

✅ Stanza loaded!

Verb Chunking Tests:
  EN before : patient should be taking medicine daily
  EN after  : patient should_be_taking medicine daily

  KOK before: ताणें दवो घेता आसा
  KOK after : ताणें दवो घेता_आसा

✅ Verb chunking ready!


## ⚙️ CELL 11 — Apply Full Preprocessing Pipeline & Save to Drive

**What this does:** Runs the complete preprocessing pipeline combining all techniques — NER protection (Vidyaapati), MWE joining (Vidyaapati), verb chunking (verb chunking paper + Papers 6 & 7) — on all 25,000 sentence pairs and saves the result to Drive.

**Smart resume:** If preprocessed data already exists in Drive from a previous session, it loads instantly and skips reprocessing entirely. This saves 15–20 minutes every time you restart Colab.

**⏱ Time:** About 15–20 minutes on first run. Instant on subsequent runs.

In [ ]:
# ── All required imports and constants ──
import os
import re
import unicodedata
import pandas as pd

DRIVE_DIR  = '/content/drive/MyDrive/Radhika_model'
TRAIN_PROC = f'{DRIVE_DIR}/data/train_processed.csv'
DEV_PROC   = f'{DRIVE_DIR}/data/dev_processed.csv'
TEST_PROC  = f'{DRIVE_DIR}/data/test_processed.csv'

# ── Reload raw splits from Drive if lost after restart ──
if 'train_df' not in dir() or not isinstance(train_df, pd.DataFrame):
    print('Reloading raw splits from Drive...')
    train_df = pd.read_csv(f'{DRIVE_DIR}/data/train.csv')
    dev_df   = pd.read_csv(f'{DRIVE_DIR}/data/dev.csv')
    test_df  = pd.read_csv(f'{DRIVE_DIR}/data/test.csv')
    print(f'✅ Reloaded — Train: {len(train_df):,}  Dev: {len(dev_df):,}  Test: {len(test_df):,}')

# ── Cleaning functions ──
def clean_english(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    contractions = {
        "n't":" not","'s":" is","'re":" are","'ve":" have",
        "'ll":" will","'d":" would","can't":"cannot",
        "won't":"will not","don't":"do not","doesn't":"does not",
        "didn't":"did not","shouldn't":"should not",
        "isn't":"is not","aren't":"are not","hasn't":"has not",
        "couldn't":"could not","wouldn't":"would not",
    }
    for c, e in contractions.items():
        text = text.replace(c, e)
    text = re.sub(r'[^a-z0-9\s\.,!?\-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_konkani(text):
    if not isinstance(text, str): return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[^\u0900-\u097F\s\.,!?।\-0-9]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── NER protection ──
DO_NOT_TRANSLATE = [
    'chia seeds','flaxseed','almond flour','coconut flour',
    'almond milk','coconut oil','olive oil','oat flour',
    'quinoa','stevia','oats','millet','sorghum',
    'baking powder','baking soda','vanilla extract',
    'vitamin a','vitamin b','vitamin c','vitamin d','vitamin e',
    'omega-3','omega-6','omega 3','omega 6',
    'zinc','magnesium','calcium','potassium',
    'antioxidant','antioxidants','probiotics','prebiotics',
    'glycemic index','insulin','triglycerides',
    'hdl','ldl','bmi','ibs','dha','epa',
]

def protect_entities(text):
    entity_map = {}
    masked = text
    for i, ent in enumerate(sorted(DO_NOT_TRANSLATE, key=len, reverse=True)):
        pattern = r'\b' + re.escape(ent) + r'\b'
        if re.search(pattern, masked, re.IGNORECASE):
            ph = f'__ENT{i}__'
            masked = re.sub(pattern, ph, masked, flags=re.IGNORECASE)
            entity_map[ph] = ent
    num_pat = r'\b(\d+\.?\d*)\s*(g|kg|ml|cal|kcal|mg|calories?|grams?|minutes?|hours?|days?)\b'
    for j, m in enumerate(re.finditer(num_pat, masked, re.IGNORECASE)):
        ph = f'__NUM{j}__'
        entity_map[ph] = m.group()
        masked = masked.replace(m.group(), ph, 1)
    return masked, entity_map

def restore_entities(text, entity_map):
    for ph, orig in entity_map.items():
        text = text.replace(ph, orig)
    return text

# ── MWE joining ──
ENGLISH_MWE = {
    'blood pressure':'blood_pressure','blood sugar':'blood_sugar',
    'blood glucose':'blood_glucose','heart rate':'heart_rate',
    'heart disease':'heart_disease','body weight':'body_weight',
    'body mass index':'body_mass_index','immune system':'immune_system',
    'digestive system':'digestive_system','healthy doughnut':'healthy_doughnut',
    'whole wheat':'whole_wheat','whole grain':'whole_grain',
    'brown sugar':'brown_sugar','low calorie':'low_calorie',
    'high fiber':'high_fiber','low fat':'low_fat',
    'sugar free':'sugar_free','gluten free':'gluten_free',
    'dairy free':'dairy_free','nutritional value':'nutritional_value',
    'dietary fiber':'dietary_fiber','saturated fat':'saturated_fat',
    'trans fat':'trans_fat','physical activity':'physical_activity',
    'weight loss':'weight_loss','weight gain':'weight_gain',
    'calorie intake':'calorie_intake','balanced diet':'balanced_diet',
    'healthy eating':'healthy_eating','meal plan':'meal_plan',
    'portion control':'portion_control','regular exercise':'regular_exercise',
    'baking powder':'baking_powder','deep fry':'deep_fry',
    'olive oil':'olive_oil','coconut oil':'coconut_oil',
}

def apply_mwe(text):
    for mwe, joined in sorted(ENGLISH_MWE.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r'\b' + re.escape(mwe) + r'\b'
        text = re.sub(pattern, joined, text, flags=re.IGNORECASE)
    return text

# ── Konkani TAM verb chunking ──
KOK_TAM = [
    (r'([\u0900-\u097F]+ता)\s+(आसा)',   r'\1_\2'),
    (r'([\u0900-\u097F]+ता)\s+(आसलो)',  r'\1_\2'),
    (r'([\u0900-\u097F]+लां)\s+(आसा)',  r'\1_\2'),
    (r'([\u0900-\u097F]+ून)\s+(घेता)',  r'\1_\2'),
    (r'([\u0900-\u097F]+ून)\s+(दिता)',  r'\1_\2'),
    (r'([\u0900-\u097F]+)\s+(जाय)',     r'\1_\2'),
    (r'([\u0900-\u097F]+)\s+(ना)',      r'\1_\2'),
    (r'([\u0900-\u097F]+तलो)\s+(आसा)', r'\1_\2'),
]

def chunk_konkani_verbs(text):
    for pat, rep in KOK_TAM:
        text = re.sub(pat, rep, text)
    return text

# ── Full preprocessing pipeline ──
def preprocess_en(text):
    text = clean_english(text)
    text, _ = protect_entities(text)
    text = apply_mwe(text)
    if 'nlp_en' in dir():
        text = chunk_english_verbs(text)
    return text

def preprocess_kok(text):
    text = clean_konkani(text)
    text = chunk_konkani_verbs(text)
    return text

def process_split(df_in, name):
    print(f'Processing {name} ({len(df_in):,} pairs)...')
    en_list, kok_list = [], []
    for idx, row in df_in.iterrows():
        en_list.append(preprocess_en(row['english']))
        kok_list.append(preprocess_kok(row['konkani']))
        if (idx + 1) % 2000 == 0:
            print(f'  → {idx+1:,} done')
    out = df_in.copy()
    out['en_proc']  = en_list
    out['kok_proc'] = kok_list
    print(f'  ✅ {name} done!')
    return out

# ── Smart resume: load from Drive if already processed ──
if os.path.exists(TRAIN_PROC):
    print('✅ Found preprocessed data in Drive — loading directly (skipping reprocessing)!')
    train_proc = pd.read_csv(TRAIN_PROC)
    dev_proc   = pd.read_csv(DEV_PROC)
    test_proc  = pd.read_csv(TEST_PROC)
else:
    train_proc = process_split(train_df, 'TRAIN')
    dev_proc   = process_split(dev_df,   'DEV')
    test_proc  = process_split(test_df,  'TEST')
    train_proc.to_csv(TRAIN_PROC, index=False)
    dev_proc.to_csv(  DEV_PROC,   index=False)
    test_proc.to_csv( TEST_PROC,  index=False)
    print(f'\n✅ Preprocessed data saved to Drive!')

print(f'\nTrain: {len(train_proc):,}  Dev: {len(dev_proc):,}  Test: {len(test_proc):,}')
print(f'\nPreprocessing example:')
print(f'  EN original  : {train_df["english"].iloc[0]}')
print(f'  EN processed : {train_proc["en_proc"].iloc[0]}')
print(f'  KOK original : {train_df["konkani"].iloc[0]}')
print(f'  KOK processed: {train_proc["kok_proc"].iloc[0]}')

✅ Found preprocessed data in Drive — loading directly (skipping reprocessing)!

Train: 19,794  Dev: 2,474  Test: 2,475

Preprocessing example:
  EN original  : rice water means the dense white water while cooking rice .
  EN processed : rice water means the dense white water while cooking rice .
  KOK original : निवळ म्हणल्यार शीत रांदतना उरिल्लें उदक .
  KOK processed: निवळ म्हणल्यार शीत रांदतना उरिल्लें उदक .


## 🔡 CELL 12 — Train Custom SentencePiece Tokenizer & Save to Drive

**What this does:** Trains a BPE tokenizer specifically on your English and Konkani health domain text.

A tokenizer converts words into numbers because neural models only process numbers. The BPE method learns the most efficient way to break your specific vocabulary into tokens — for example learning that 'blood_pressure' should be one token, that common Devanagari combinations should be single tokens etc. Vocabulary size 8,000 is optimal for a low-resource language — large enough to cover your health vocabulary, small enough that every token appears frequently enough in training to be learned well.

Character coverage 0.9999 means it covers 99.99% of all characters — essential for Devanagari script which has many characters.

**Smart resume:** If tokenizer already saved in Drive, loads instantly.

In [ ]:
import os
import shutil
import sentencepiece as spm
import pandas as pd

DRIVE_DIR  = '/content/drive/MyDrive/Radhika_model'
TOK_MODEL  = f'{DRIVE_DIR}/tokenizer/en_kok_health'

# ── Reload preprocessed data if lost after restart ──
if 'train_proc' not in dir() or not isinstance(train_proc, pd.DataFrame):
    print('Reloading preprocessed data from Drive...')
    train_proc = pd.read_csv(f'{DRIVE_DIR}/data/train_processed.csv')
    dev_proc   = pd.read_csv(f'{DRIVE_DIR}/data/dev_processed.csv')
    test_proc  = pd.read_csv(f'{DRIVE_DIR}/data/test_processed.csv')
    print(f'✅ Reloaded — Train: {len(train_proc):,}  Dev: {len(dev_proc):,}  Test: {len(test_proc):,}')

# ── Smart resume: load tokenizer from Drive if already trained ──
if os.path.exists(f'{TOK_MODEL}.model'):
    print('✅ Found tokenizer in Drive — loading directly!')
else:
    print('Training SentencePiece tokenizer...')
    with open('/content/tok_input.txt', 'w', encoding='utf-8') as f:
        for text in list(train_proc['en_proc']) + list(train_proc['kok_proc']):
            if str(text).strip():
                f.write(str(text).strip() + '\n')

    spm.SentencePieceTrainer.train(
        input              = '/content/tok_input.txt',
        model_prefix       = '/content/en_kok_health',
        vocab_size         = 8000,
        character_coverage = 0.9999,
        model_type         = 'bpe',
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        user_defined_symbols = '<health>,<medicine>,<diet>,<doughnut>'
    )
    shutil.copy('/content/en_kok_health.model', f'{TOK_MODEL}.model')
    shutil.copy('/content/en_kok_health.vocab', f'{TOK_MODEL}.vocab')
    print('✅ Tokenizer saved to Drive!')

# ── Load tokenizer ──
sp = spm.SentencePieceProcessor()
sp.load(f'{TOK_MODEL}.model')

test_en  = 'the patient should_be_taking healthy_doughnut daily'
test_kok = 'रोगी_न दर दीस तब्येतीक डोनट खावचो'
print(f'Vocab size  : {sp.get_piece_size()}')
print(f'EN tokens   : {sp.encode(test_en,  out_type=str)}')
print(f'KOK tokens  : {sp.encode(test_kok, out_type=str)}')
print('\n✅ Tokenizer ready!')

✅ Found tokenizer in Drive — loading directly!
Vocab size  : 8000
EN tokens   : ['▁the', '▁patient', '▁should', '_', 'be', '_', 't', 'aking', '▁healthy', '_', 'd', 'ough', 'nut', '▁daily']
KOK tokens  : ['▁रोग', 'ी', '_', 'न', '▁दर', '▁दीस', '▁त', 'ब', '्य', 'े', 'त', 'ीक', '▁ड', 'ोन', 'ट', '▁खाव', 'चो']

✅ Tokenizer ready!


## 🤖 CELL 13 — Load IndicTrans2 (Paper 5 — Gala et al. 2023)

**What this does:** Downloads and loads IndicTrans2 — the model from Paper 5 (Gala et al. 2023). This is the state-of-the-art model for Indian language translation built by AI4Bharat at IIT Madras.

IndicTrans2 was pre-trained on massive multilingual data covering all 22 scheduled Indian languages including Konkani. This means before you even fine-tune it, it already knows how Konkani words and grammar work. Fine-tuning then specializes it for your health domain.

**Smart resume:** If the final model already exists in your Drive from a previous completed training, it loads from Drive instead of downloading again.

**⏱ Time:** 5–10 minutes to download (~800MB).

In [ ]:
# ── Load model ──
if os.path.exists(f'{FINAL_MODEL}/config.json'):
    print('✅ Found final model in Drive — loading from Drive!')

    hf_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        token=hf_token,      # ← comma was missing here
        use_fast=False
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        FINAL_MODEL,
        trust_remote_code=True
    )
else:
    print(f'Downloading {MODEL_NAME}...')
    hf_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        token=hf_token,
        use_fast=False
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        token=hf_token
    )

model = model.to(device)
print(f'✅ Model and Tokenizer loaded successfully on {device}!')

✅ Found final model in Drive — loading from Drive!
✅ Model and Tokenizer loaded successfully on cuda!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from huggingface_hub import login
from google.colab import userdata

# ── Constants ──
DRIVE_DIR   = '/content/drive/MyDrive/Radhika_model'
MODEL_NAME  = 'ai4bharat/indictrans2-en-indic-dist-200M'
FINAL_MODEL = f'{DRIVE_DIR}/final_model'
CKPT_DIR    = f'{DRIVE_DIR}/checkpoints'
SRC_LANG    = 'eng_Latn'
TGT_LANG    = 'gom_Deva'
MAX_LEN     = 128
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load token ──
hf_token = userdata.get('HF_TOKEN')
print('✅ Token loaded successfully!')

# ── Login ──
login(token=hf_token, add_to_git_credential=False)
print('✅ Logged in to HuggingFace!')

# ── Reload preprocessed data ──
if 'train_proc' not in dir() or not isinstance(train_proc, pd.DataFrame):
    print('Reloading preprocessed data from Drive...')
    train_proc = pd.read_csv(f'{DRIVE_DIR}/data/train_processed.csv')
    dev_proc   = pd.read_csv(f'{DRIVE_DIR}/data/dev_processed.csv')
    test_proc  = pd.read_csv(f'{DRIVE_DIR}/data/test_processed.csv')
    print(f'✅ Reloaded — Train: {len(train_proc):,}  Dev: {len(dev_proc):,}  Test: {len(test_proc):,}')

# ── FIXED: Always load tokenizer from HuggingFace ONLY ──
print('Loading tokenizer from HuggingFace...')
hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=hf_token
)
print('✅ Tokenizer loaded!')

# ── Load model from Drive ──
import shutil
LOCAL_MODEL = '/content/final_model_local'
if os.path.exists(LOCAL_MODEL):
    shutil.rmtree(LOCAL_MODEL)
print('Copying model from Drive...')
shutil.copytree(FINAL_MODEL, LOCAL_MODEL)

print('Loading model...')
model = AutoModelForSeq2SeqLM.from_pretrained(
    LOCAL_MODEL,
    trust_remote_code=True,
    ignore_mismatched_sizes=True
)
model = model.to(device)
total = sum(p.numel() for p in model.parameters())

print(f'\n✅ Model loaded successfully!')
print(f'   Parameters : {total:,}')
print(f'   Device     : {device}')
print(f'   Model      : {MODEL_NAME}')

✅ Token loaded successfully!
✅ Logged in to HuggingFace!
Loading tokenizer from HuggingFace...
✅ Tokenizer loaded!
Copying model from Drive...
Loading model...

✅ Model loaded successfully!
   Parameters : 211,776,512
   Device     : cuda
   Model      : ai4bharat/indictrans2-en-indic-dist-200M


## 📊 CELL 14 — Tokenize Dataset (Cached for Fast Resume)

**What this does:** Converts all your preprocessed sentence pairs into numerical format for training. Every sentence gets tokenized — broken into tokens and mapped to numbers. The English input gets the language direction tag `>>kok_Deva<<` which tells IndicTrans2 to translate into Konkani Devanagari script.

**Smart caching:** Saves the tokenized dataset to Drive. On any restart, loads in seconds instead of re-tokenizing.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_from_disk
from transformers import AutoTokenizer
from google.colab import userdata
from huggingface_hub import login

# ── Constants ──
DRIVE_DIR      = '/content/drive/MyDrive/Radhika_model'
TRAIN_TOK_PATH = f'{DRIVE_DIR}/data/train_tokenized'
DEV_TOK_PATH   = f'{DRIVE_DIR}/data/dev_tokenized'
MODEL_NAME     = 'ai4bharat/indictrans2-en-indic-dist-200M'
MAX_LEN        = 128
SRC_LANG       = 'eng_Latn'
TGT_LANG       = 'gom_Deva'

# ── Login ──
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
except:
    hf_token = None

# ── Load tokenizer from HuggingFace ONLY ──
print('Loading tokenizer from HuggingFace...')
hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=hf_token
)
print('✅ Tokenizer loaded!')

# ── Reload preprocessed data ──
print('Loading preprocessed data...')
train_proc = pd.read_csv(f'{DRIVE_DIR}/data/train_processed.csv')
dev_proc   = pd.read_csv(f'{DRIVE_DIR}/data/dev_processed.csv')
test_proc  = pd.read_csv(f'{DRIVE_DIR}/data/test_processed.csv')
print(f'✅ Loaded — Train: {len(train_proc):,} Dev: {len(dev_proc):,}')

# ── Test tokenization first ──
print('\nTesting correct tag format...')
test_input = f'{SRC_LANG} {TGT_LANG} eat healthy food daily'
test_out   = hf_tokenizer(test_input, max_length=MAX_LEN, truncation=True)
print(f'✅ Tag format correct! Tokens: {len(test_out["input_ids"])}')

# ── Tokenization function ──
def tokenize_pairs(examples):
    inputs  = [f'{SRC_LANG} {TGT_LANG} ' + str(t) for t in examples['en_proc']]
    targets = [str(t) for t in examples['kok_proc']]
    model_inputs = hf_tokenizer(
        inputs,
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length'
    )
    labels = hf_tokenizer(
        text_target=targets,
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length'
    )
    model_inputs['labels'] = [
        [(t if t != hf_tokenizer.pad_token_id else -100) for t in lbl]
        for lbl in labels['input_ids']
    ]
    return model_inputs

# ── Tokenize ──
print('\nTokenizing all data fresh...')
train_hf  = Dataset.from_pandas(train_proc[['en_proc','kok_proc']].astype(str))
dev_hf    = Dataset.from_pandas(dev_proc[['en_proc','kok_proc']].astype(str))
train_tok = train_hf.map(
    tokenize_pairs, batched=True, batch_size=128,
    remove_columns=train_hf.column_names
)
dev_tok = dev_hf.map(
    tokenize_pairs, batched=True, batch_size=128,
    remove_columns=dev_hf.column_names
)
train_tok.save_to_disk(TRAIN_TOK_PATH)
dev_tok.save_to_disk(DEV_TOK_PATH)
print(f'\n✅ Tokenized and saved!')
print(f'Train: {len(train_tok):,}  Dev: {len(dev_tok):,}')
print('✅ Now run Step 3!')

Loading tokenizer from HuggingFace...
✅ Tokenizer loaded!
Loading preprocessed data...
✅ Loaded — Train: 19,794 Dev: 2,474

Testing correct tag format...
✅ Tag format correct! Tokens: 7

Tokenizing all data fresh...


Map:   0%|          | 0/19794 [00:00<?, ? examples/s]

Map:   0%|          | 0/2474 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/19794 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2474 [00:00<?, ? examples/s]


✅ Tokenized and saved!
Train: 19,794  Dev: 2,474
✅ Now run Step 3!


## 🏋️ CELL 15 — FINE-TUNE MODEL (Optimized T4 + Auto-Resume + Save to Drive)

**What this does:** This is the main training cell. Fine-tunes IndicTrans2 on your 20,000 health domain sentence pairs.

**Speed optimizations for T4:**
- `fp16=True` — uses 16-bit precision, makes training 2x faster on T4
- `batch_size=32` — larger batches = more efficient GPU use
- `gradient_accumulation=2` — effective batch of 64 without extra memory
- `cosine learning rate` — gradually reduces LR for better final BLEU
- `dataloader_num_workers=2` — loads data in parallel while GPU trains

**Auto-resume:** Automatically detects the last checkpoint in your Drive and continues from that exact point. You never lose training progress.

**Auto-save:** Saves a checkpoint every 500 steps to Drive. If Colab crashes you lose at most 500 steps worth of training.

**⏱ Expected time on T4:** ~25–35 min per epoch. 15 epochs = ~7–8 hours total. Early stopping will likely stop it around epoch 10–12.

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.serialization
import numpy.core.multiarray
import pandas as pd
from datasets import load_from_disk
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from huggingface_hub import login
from google.colab import userdata
import evaluate as hf_evaluate
import gc

# ══════════════════════════════════════════
# FIX 1 — PyTorch 2.6 checkpoint fix
# ══════════════════════════════════════════
torch.serialization.add_safe_globals([
    numpy.core.multiarray._reconstruct,
    numpy.ndarray,
    numpy.dtype,
])

# ══════════════════════════════════════════
# FIX 2 — Patch torch.load for full safety
# ══════════════════════════════════════════
_original_torch_load = torch.load
def _patched_torch_load(f, *args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(f, *args, **kwargs)
torch.load = _patched_torch_load

# ── Constants ──
DRIVE_DIR   = '/content/drive/MyDrive/Radhika_model'
FINAL_MODEL = f'{DRIVE_DIR}/final_model'
CKPT_DIR    = f'{DRIVE_DIR}/checkpoints'
MODEL_NAME  = 'ai4bharat/indictrans2-en-indic-dist-200M'
MAX_LEN     = 128
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gc.collect()
torch.cuda.empty_cache()
print(f'GPU free memory: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

# ── Login ──
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('✅ HuggingFace logged in!')
except:
    hf_token = None

# ── Load tokenizer from HuggingFace ONLY ──
print('Loading tokenizer...')
hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=hf_token
)
print('✅ Tokenizer loaded!')

# ── Find last checkpoint ──
CKPT_LIST = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
last_ckpt  = CKPT_LIST[-1] if CKPT_LIST else None

if last_ckpt:
    print(f'✅ Found checkpoint: {last_ckpt}')
    print('Loading model from checkpoint...')
    model = AutoModelForSeq2SeqLM.from_pretrained(
        last_ckpt,
        trust_remote_code=True,
        ignore_mismatched_sizes=True
    )
else:
    print('No checkpoint found — loading fresh base model...')
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        token=hf_token
    )

model = model.to(device)
print('✅ Model loaded!')

# ── Load tokenized data ──
TRAIN_TOK_PATH = f'{DRIVE_DIR}/data/train_tokenized'
DEV_TOK_PATH   = f'{DRIVE_DIR}/data/dev_tokenized'
print('Loading tokenized data...')
train_tok = load_from_disk(TRAIN_TOK_PATH)
dev_tok   = load_from_disk(DEV_TOK_PATH)
print(f'✅ Train: {len(train_tok):,}  Dev: {len(dev_tok):,}')

# ── Metrics ──
bleu_metric = hf_evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    decoded_preds  = hf_tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, hf_tokenizer.pad_token_id)
    decoded_labels = hf_tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]
    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {'bleu': round(result['score'], 2)}

# ── Training arguments ──
training_args = Seq2SeqTrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = 8,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 8,
    warmup_steps                = 200,
    weight_decay                = 0.01,
    logging_steps               = 100,
    eval_strategy               = 'steps',
    eval_steps                  = 500,
    save_strategy               = 'steps',
    save_steps                  = 500,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    predict_with_generate       = True,
    generation_max_length       = MAX_LEN,
    fp16                        = True,
    learning_rate               = 5e-5,
    dataloader_num_workers      = 1,
    report_to                   = 'none',
    optim                       = 'adamw_torch',
    lr_scheduler_type           = 'cosine',
    generation_num_beams        = 4,
    gradient_checkpointing      = True,
    dataloader_pin_memory       = False,
)

# ── Data collator ──
data_collator = DataCollatorForSeq2Seq(
    hf_tokenizer, model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

# ── Trainer ──
trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = dev_tok,
    tokenizer       = hf_tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
)

# ── Resume from checkpoint ──
if last_ckpt:
    print(f'\n🔄 Resuming from: {last_ckpt}')
else:
    print('\n🚀 Starting fresh training...')

print(f'  Epochs     : 8')
print(f'  Batch size : 8 (effective 64)')
print(f'  Device     : {device}')
print(f'  Saves every 500 steps automatically')
print()

trainer.train(resume_from_checkpoint=last_ckpt)

# ── Restore original torch.load ──
torch.load = _original_torch_load

# ── Save best model ──
trainer.save_model(FINAL_MODEL)
hf_tokenizer.save_pretrained(FINAL_MODEL)
print(f'\n✅ Training complete!')
print(f'✅ Best model saved to: {FINAL_MODEL}')

# ══════════════════════════════════════════
# IMMEDIATE TRANSLATION TEST AFTER TRAINING
# ══════════════════════════════════════════
print('\n' + '='*50)
print('TRANSLATION TEST — checking model quality')
print('='*50)

SRC_LANG = 'eng_Latn'
TGT_LANG = 'gom_Deva'

def translate_test(text):
    inputs = hf_tokenizer(
        f'{SRC_LANG} {TGT_LANG} ' + text.lower().strip(),
        return_tensors='pt',
        max_length=MAX_LEN,
        truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=4,
            max_length=MAX_LEN,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return hf_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

test_sentences = [
    'eating healthy food is good for your body',
    'avoid deep fried foods if you have high blood pressure',
    'a balanced diet helps maintain a healthy body weight',
]

for s in test_sentences:
    print(f'\nEN  : {s}')
    print(f'KOK : {translate_test(s)}')

print('\n✅ If above shows proper Konkani — training successful!')

GPU free memory: 14.43 GB
✅ HuggingFace logged in!
Loading tokenizer...
✅ Tokenizer loaded!
✅ Found checkpoint: /content/drive/MyDrive/Radhika_model/checkpoints/checkpoint-2472
Loading model from checkpoint...
✅ Model loaded!
Loading tokenized data...
✅ Train: 19,794  Dev: 2,474



🔄 Resuming from: /content/drive/MyDrive/Radhika_model/checkpoints/checkpoint-2472
  Epochs     : 8
  Batch size : 8 (effective 64)
  Device     : cuda
  Saves every 500 steps automatically



There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Step,Training Loss,Validation Loss



✅ Training complete!
✅ Best model saved to: /content/drive/MyDrive/Radhika_model/final_model

TRANSLATION TEST — checking model quality

EN  : eating healthy food is good for your body


KOK : भलायकेवंत जेवण जेवण तुमचे कुडी खातीर बरें आसता.

EN  : avoid deep fried foods if you have high blood pressure
KOK : जर तुमकां उच्च रक्तदाब आसत जाल्यार तळिल्ले तळिल्ले जिनस टाळचे.

EN  : a balanced diet helps maintain a healthy body weight
KOK : संतुळीत आहार भलायकेवंत कुडीचें वजन तिगोवन दवरता.

✅ If above shows proper Konkani — training successful!


## 📐 CELL 16 — Full Evaluation: BLEU + chrF + Back-Translation (Papers 2, 3, 4)

**What this does:** Runs complete evaluation on your hidden test set using three methods from three different papers.

**BLEU score (Paper 4 — Neves 2023):** Measures word-level overlap between your system output and reference translations. For English→Konkani health domain with 25K sentences, target is 25–40.

**chrF score (Paper 4 — Neves 2023):** Measures character-level similarity. More reliable for Devanagari script because Konkani words change form with tense, gender and number — chrF recognizes similarity even when exact words differ.

**Back-translation quality check (Paper 3 — Patil & Davies 2014):** Translates your Konkani output back to English and compares with the original. If they match, translation is likely correct. This is the method Patil & Davies used to evaluate Google Translate safety.

All results are saved to your Drive.

In [ ]:
import os
import re
import json
import torch
import numpy as np
import pandas as pd
import gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import BLEU, CHRF
from huggingface_hub import login
from google.colab import userdata

# ── Constants ──
DRIVE_DIR   = '/content/drive/MyDrive/Radhika_model'
FINAL_MODEL = '/content/drive/MyDrive/Radhika_model/checkpoints/checkpoint-2472'
MODEL_NAME  = 'ai4bharat/indictrans2-en-indic-dist-200M'
SRC_LANG    = 'eng_Latn'
TGT_LANG    = 'gom_Deva'
MAX_LEN     = 128
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gc.collect()
torch.cuda.empty_cache()

# ── Reload tokenizer if lost ──
if 'hf_tokenizer' not in dir():
    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token, add_to_git_credential=False)
    except:
        pass
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
print('✅ Tokenizer reloaded!')

# ── Reload model if lost ──
if 'model' not in dir():
    model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL, trust_remote_code=True)
    model = model.to(device)
    print('✅ Model reloaded!')

# ── Reload test data if lost ──
if 'test_proc' not in dir() or not isinstance(test_proc, pd.DataFrame):
    test_proc = pd.read_csv(f'{DRIVE_DIR}/data/test_processed.csv')
    print(f'✅ Test data reloaded: {len(test_proc):,} pairs')

if 'test_df' not in dir() or not isinstance(test_df, pd.DataFrame):
    test_df = pd.read_csv(f'{DRIVE_DIR}/data/test.csv')

# ── Helper functions ──
DO_NOT_TRANSLATE = [
    'chia seeds','flaxseed','almond flour','coconut flour',
    'almond milk','coconut oil','olive oil','oat flour',
    'quinoa','stevia','oats','millet','sorghum',
    'baking powder','baking soda','vanilla extract',
    'vitamin a','vitamin b','vitamin c','vitamin d','vitamin e',
    'omega-3','omega-6','omega 3','omega 6',
    'zinc','magnesium','calcium','potassium',
    'antioxidant','antioxidants','probiotics','prebiotics',
    'glycemic index','insulin','triglycerides',
    'hdl','ldl','bmi','ibs','dha','epa',
]

def clean_english(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    contractions = {
        "n't":" not","'s":" is","'re":" are","'ve":" have",
        "'ll":" will","'d":" would","can't":"cannot",
        "won't":"will not","don't":"do not","doesn't":"does not",
        "didn't":"did not","shouldn't":"should not",
        "isn't":"is not","aren't":"are not","hasn't":"has not",
        "couldn't":"could not","wouldn't":"would not",
    }
    for c, e in contractions.items():
        text = text.replace(c, e)
    text = re.sub(r'[^a-z0-9\s\.,!?\-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def protect_entities(text):
    entity_map = {}
    masked = text
    for i, ent in enumerate(sorted(DO_NOT_TRANSLATE, key=len, reverse=True)):
        pattern = r'\b' + re.escape(ent) + r'\b'
        if re.search(pattern, masked, re.IGNORECASE):
            ph = f'__ENT{i}__'
            masked = re.sub(pattern, ph, masked, flags=re.IGNORECASE)
            entity_map[ph] = ent
    num_pat = r'\b(\d+\.?\d*)\s*(g|kg|ml|cal|kcal|mg|calories?|grams?|minutes?|hours?|days?)\b'
    for j, m in enumerate(re.finditer(num_pat, masked, re.IGNORECASE)):
        ph = f'__NUM{j}__'
        entity_map[ph] = m.group()
        masked = masked.replace(m.group(), ph, 1)
    return masked, entity_map

def restore_entities(text, entity_map):
    for ph, orig in entity_map.items():
        text = text.replace(ph, orig)
    return text

ENGLISH_MWE = {
    'blood pressure':'blood_pressure','blood sugar':'blood_sugar',
    'heart rate':'heart_rate','heart disease':'heart_disease',
    'body weight':'body_weight','immune system':'immune_system',
    'whole wheat':'whole_wheat','whole grain':'whole_grain',
    'low calorie':'low_calorie','high fiber':'high_fiber',
    'low fat':'low_fat','sugar free':'sugar_free',
    'weight loss':'weight_loss','weight gain':'weight_gain',
    'balanced diet':'balanced_diet','meal plan':'meal_plan',
    'blood glucose':'blood_glucose','body mass index':'body_mass_index',
}

def apply_mwe(text):
    for mwe, joined in sorted(ENGLISH_MWE.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r'\b' + re.escape(mwe) + r'\b'
        text = re.sub(pattern, joined, text, flags=re.IGNORECASE)
    return text

def translate_sentence(text):
    proc = clean_english(text)
    proc, emap = protect_entities(proc)
    proc = apply_mwe(proc)
    inputs = hf_tokenizer(
        f'{SRC_LANG} {TGT_LANG} ' + proc,
        return_tensors='pt', max_length=MAX_LEN, truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=4,
            max_length=MAX_LEN,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
        )
    out = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)
    out = out.replace('_', ' ')
    out = restore_entities(out, emap)
    out = re.sub(r'\s([।,?!])', r'\1', out).strip()
    return out

# ── Run evaluation ──
bleu_scorer = BLEU()
chrf_scorer = CHRF()

print('Evaluating on test set...')
hypotheses, references = [], []
for i, row in test_proc.iterrows():
    hypotheses.append(translate_sentence(row['english']))
    references.append(str(row['konkani']))
    if len(hypotheses) % 500 == 0:
        print(f'  → {len(hypotheses):,} / {len(test_proc):,} done')

bleu = bleu_scorer.corpus_score(hypotheses, [references])
chrf = chrf_scorer.corpus_score(hypotheses, [references])

print('\n' + '='*55)
print('      FINAL EVALUATION RESULTS')
print('='*55)
print(f'  BLEU Score : {bleu.score:.2f}')
print(f'  chrF Score : {chrf.score:.2f}')
print('='*55)

print('\nSample translations:')
for i in range(5):
    print(f'\n  [{i+1}] EN  : {test_df["english"].iloc[i]}')
    print(f'       KOK : {hypotheses[i]}')
    print(f'       REF : {references[i]}')

# ── Save results ──
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
results = {
    'bleu': bleu.score,
    'chrf': chrf.score,
    'test_size': len(test_proc),
    'train_size': 19794
}
with open(f'{DRIVE_DIR}/results/evaluation.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\n✅ Results saved to Drive!')

✅ Tokenizer reloaded!
Evaluating on test set...
  → 500 / 2,475 done
  → 1,000 / 2,475 done
  → 1,500 / 2,475 done
  → 2,000 / 2,475 done

      FINAL EVALUATION RESULTS
  BLEU Score : 21.85
  chrF Score : 54.67

Sample translations:

  [1] EN  : in this way the blood loss stops even without surgery .
       KOK : हे तरेन ओपेरासांवा बगर लेगीत रक्तस्राव थांबता.
       REF : हे तरेन ओपेरासांवा बगरूच रगत व्हांवपाचें बंद जाता .

  [2] EN  : pruritis - itch or itching appears in many diseases like - dermatitis , prygo eczema , sheetapitta , louse production , shingles , white discharge , occurrence of stomach germs or of diabetes .
       KOK : खोरोज - कितल्याश्याच दुयेंसांनी खाज वा खाज येता जशें - डर्मेटायटीस, प्रॅगो एक्जिमा, शेटापित्ता, उयेची निर्मणी, दादलेपण, धवो स्राव, पोटाचे जिवाणू जावप वा गोडेंमूत.
       REF : खाज - खाज वा खोरोज जायत्या दुयेंसांनी प्रकट जातात जशीं - कातीची दूख , प्ररायगो खोरोज , प्ररायगो खोरोज , थंड पित्त , उयो जावप , चरतें , धवें वचप , चुन्ने जावप वा गोड्यामुताचें ज

In [ ]:
# DIAGNOSTIC CELL — Run this first
from transformers import AutoTokenizer
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)

MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)

# Check what language codes the tokenizer actually knows
print("All special tokens:")
print(hf_tokenizer.all_special_tokens[:50])

print("\nLooking for Konkani/Gom tokens:")
vocab = hf_tokenizer.get_vocab()
gom_tokens = [t for t in vocab.keys() if 'gom' in t.lower() or 'kok' in t.lower()]
print(f"Found: {gom_tokens}")

All special tokens:
['<s>', '</s>', '<unk>', '<pad>']

Looking for Konkani/Gom tokens:
Found: ['gom_Deva', '▁Kok', '▁Bangkok', '▁Gom', '▁Gomez', '▁Montgomery', 'kok']


## 🌿 CELL 17 — Sample Translations: See 10 Outputs

**What this does:** Shows you 10 sample translations from your trained model on the test set so you can visually judge the quality of the Konkani output. This is the human evaluation component recommended by Paper 2 (Ugas 2025) — you should check accuracy, natural language, and medical correctness alongside the automatic BLEU score.

In [ ]:
print('📋 SAMPLE TRANSLATIONS — Visual Quality Check (Paper 2 — Ugas 2025)')
print('='*65)
for i in range(10):
    en  = test_df['english'].iloc[i]
    ref = test_df['konkani'].iloc[i]
    sys = translate_sentence(en)
    print(f'\n[{i+1}]')
    print(f'  English   : {en}')
    print(f'  Reference : {ref}')
    print(f'  System    : {sys}')
print('\n✅ Visual inspection done!')
print('Check: Is the medical meaning correct? Is the Konkani natural?')

📋 SAMPLE TRANSLATIONS — Visual Quality Check (Paper 2 — Ugas 2025)

[1]
  English   : in this way the blood loss stops even without surgery .
  Reference : हे तरेन ओपेरासांवा बगरूच रगत व्हांवपाचें बंद जाता .
  System    : हे तरेन ओपेरासांवा बगर लेगीत रगत वचपाचें बंद जाता.

[2]
  English   : pruritis - itch or itching appears in many diseases like - dermatitis , prygo eczema , sheetapitta , louse production , shingles , white discharge , occurrence of stomach germs or of diabetes .
  Reference : खाज - खाज वा खोरोज जायत्या दुयेंसांनी प्रकट जातात जशीं - कातीची दूख , प्ररायगो खोरोज , प्ररायगो खोरोज , थंड पित्त , उयो जावप , चरतें , धवें वचप , चुन्ने जावप वा गोड्यामुताचें जावप .
  System    : मुत्रपिंड - खूबश्या दुयेंसांनी खाजी वा खाज येता जशें - डर्मेटायटीस, प्रॅगो एक्जिमा, शितापिट्टा, उयांची निर्मणी, चरतें, धवें वचप, पोटांत जिवाणू जावप वा गोड्यामुताचें.

[3]
  English   : kids getting scared at nights night terrors - watching horrible scenes in the day , having worms in the intestine , com

In [ ]:
import os
import re
import torch
import pandas as pd
import gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login

# ── Constants ──
DRIVE_DIR   = '/content/drive/MyDrive/Radhika_model'
FINAL_MODEL = f'{DRIVE_DIR}/final_model'
SRC_LANG    = 'eng_Latn'
TGT_LANG    = 'gom_Deva'
MAX_LEN     = 128
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gc.collect()
torch.cuda.empty_cache()

# ── Reload if lost ──
if 'hf_tokenizer' not in dir():
    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token, add_to_git_credential=False)
    except:
        pass
    hf_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL, trust_remote_code=True)

if 'model' not in dir():
    model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL, trust_remote_code=True)
    model = model.to(device)

if 'test_df' not in dir():
    test_df = pd.read_csv(f'{DRIVE_DIR}/data/test.csv')

def translate_sentence(text):
    inputs = hf_tokenizer(
        f'{SRC_LANG} {TGT_LANG} ' + text.lower().strip(),
        return_tensors='pt', max_length=MAX_LEN, truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, num_beams=4, max_length=MAX_LEN,
            early_stopping=True, no_repeat_ngram_size=3,
        )
    return hf_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print('📋 SAMPLE TRANSLATIONS — Visual Quality Check')
print('='*65)
for i in range(10):
    en  = test_df['english'].iloc[i]
    ref = test_df['konkani'].iloc[i]
    sys = translate_sentence(en)
    print(f'\n[{i+1}]')
    print(f'  English   : {en}')
    print(f'  Reference : {ref}')
    print(f'  System    : {sys}')

print('\n✅ Visual inspection done!')

📋 SAMPLE TRANSLATIONS — Visual Quality Check

[1]
  English   : in this way the blood loss stops even without surgery .
  Reference : हे तरेन ओपेरासांवा बगरूच रगत व्हांवपाचें बंद जाता .
  System    : हे तरेन ओपेरासांवा बगर लेगीत रगत व्हांवप बंद जाता.

[2]
  English   : pruritis - itch or itching appears in many diseases like - dermatitis , prygo eczema , sheetapitta , louse production , shingles , white discharge , occurrence of stomach germs or of diabetes .
  Reference : खाज - खाज वा खोरोज जायत्या दुयेंसांनी प्रकट जातात जशीं - कातीची दूख , प्ररायगो खोरोज , प्ररायगो खोरोज , थंड पित्त , उयो जावप , चरतें , धवें वचप , चुन्ने जावप वा गोड्यामुताचें जावप .
  System    : प्रुरायटीस - कितल्याश्याच दुयेंसांनी जशें - डर्मॅटायटीस, प्रॅगो एक्जिमा, शीतपित्त, उयांची उत्पत्ती, चरतें, धवें वचप, पोटांत जिवाणू जावप वा गोडेंमूत हातूंत खाज वा खाज येता.

[3]
  English   : kids getting scared at nights night terrors - watching horrible scenes in the day , having worms in the intestine , coming out of teeth

## ✏️ CELL 18 — Translate Any New English Sentence

**What this does:** Your live translation tool. Add any English health sentence to the list and run the cell to see the Konkani translation from your trained model. This is how you would demonstrate your system working.

In [ ]:
# ✏️ Add your own sentences here to test
my_sentences = [
    'A healthy doughnut made with whole wheat flour is good for diabetic patients.',
    'You should eat nutritious food and exercise daily.',
    'Avoid deep fried foods if you have high blood pressure.',
    'A sugar free doughnut is a healthy alternative for weight loss.',
    'Include fiber rich ingredients in your daily meal plan.',
    'Blood sugar levels can be controlled through a balanced diet.',
    'The baked doughnut contains only 150 calories per serving.',
    'This recipe uses whole wheat flour and is low calorie.',
    'Eating a nutritious breakfast improves your immune system.',
    'A balanced diet helps maintain a healthy body weight.',
]

print('🌿 ENGLISH → KONKANI HEALTH TRANSLATION SYSTEM')
print('='*65)
for i, sentence in enumerate(my_sentences, 1):
    result = translate_sentence(sentence)
    print(f'\n[{i}] EN  : {sentence}')
    print(f'     KOK : {result}')

print('\n✅ Translation complete!')
print('💡 Edit my_sentences above to translate your own health sentences.')

In [ ]:
import os
import torch
import gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login

# ── Constants ──
DRIVE_DIR   = '/content/drive/MyDrive/Radhika_model'
FINAL_MODEL = f'{DRIVE_DIR}/final_model'
SRC_LANG    = 'eng_Latn'
TGT_LANG    = 'gom_Deva'
MAX_LEN     = 128
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gc.collect()
torch.cuda.empty_cache()

# ── Reload if lost ──
if 'hf_tokenizer' not in dir():
    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token, add_to_git_credential=False)
    except:
        pass
    hf_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL, trust_remote_code=True)

if 'model' not in dir():
    model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL, trust_remote_code=True)
    model = model.to(device)

def translate_sentence(text):
    inputs = hf_tokenizer(
        f'{SRC_LANG} {TGT_LANG} ' + text.lower().strip(),
        return_tensors='pt', max_length=MAX_LEN, truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, num_beams=4, max_length=MAX_LEN,
            early_stopping=True, no_repeat_ngram_size=3,
        )
    return hf_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

my_sentences = [
    'A healthy doughnut made with whole wheat flour is good for diabetic patients.',
    'You should eat nutritious food and exercise daily.',
    'Avoid deep fried foods if you have high blood pressure.',
    'A sugar free doughnut is a healthy alternative for weight loss.',
    'Include fiber rich ingredients in your daily meal plan.',
    'Blood sugar levels can be controlled through a balanced diet.',
    'The baked doughnut contains only 150 calories per serving.',
    'This recipe uses whole wheat flour and is low calorie.',
    'Eating a nutritious breakfast improves your immune system.',
    'A balanced diet helps maintain a healthy body weight.',
]

print('🌿 ENGLISH → KONKANI HEALTH TRANSLATION SYSTEM')
print('='*65)
for i, sentence in enumerate(my_sentences, 1):
    result = translate_sentence(sentence)
    print(f'\n[{i}] EN  : {sentence}')
    print(f'     KOK : {result}')

print('\n✅ Translation complete!')

🌿 ENGLISH → KONKANI HEALTH TRANSLATION SYSTEM

[1] EN  : A healthy doughnut made with whole wheat flour is good for diabetic patients.
     KOK : गोड्यामुताच्या दुयेंतीं खातीर एक भलायकेवंत पिठ्यान तयार केल्लो डोनट बरो आसता.

[2] EN  : You should eat nutritious food and exercise daily.
     KOK : पुश्टीक आहार आनी व्यायाम सदांच करपाक_जाय.

[3] EN  : Avoid deep fried foods if you have high blood pressure.
     KOK : जर तुमकां उच्च रक्तदाब आसा तर तळिल्लें खावप टाळचें.

[4] EN  : A sugar free doughnut is a healthy alternative for weight loss.
     KOK : साकर_नाशिल्लें डोनट वजन उणें करपा खातीर एक भलायकेवंत उपाय.

[5] EN  : Include fiber rich ingredients in your daily meal plan.
     KOK : तुमच्या दीसपट्ट्या जेवणांत तंतूमय तत्वांचो आसपाव करात.

[6] EN  : Blood sugar levels can be controlled through a balanced diet.
     KOK : संतुळीत आहारा वरवीं रक्तांतले साकरीचो स्तर नियंत्रीत करूं येता.

[7] EN  : The baked doughnut contains only 150 calories per serving.
     KOK : बेक केल्ल्या डोनटांत दर 

## 💾 CELL 19 — Confirm Everything Is Saved in Drive

**What this does:** Final check that confirms all files are properly saved to your `Radhika_model` folder in Google Drive. Lists everything that was saved so you can verify nothing is missing.

In [ ]:
import os

DRIVE_DIR = '/content/drive/MyDrive/Radhika_model'

print('📁 RADHIKA_MODEL — Contents in Google Drive')
print('='*50)
for root, dirs, files in os.walk(DRIVE_DIR):
    level     = root.replace(DRIVE_DIR, '').count(os.sep)
    indent    = '  ' * level
    folder    = os.path.basename(root)
    if level == 0:
        print(f'📁 Radhika_model/')
    else:
        print(f'{indent}📁 {folder}/')
    subindent = '  ' * (level + 1)
    for file in files:
        fpath = os.path.join(root, file)
        fsize = os.path.getsize(fpath) / 1e6
        print(f'{subindent}📄 {file}  ({fsize:.1f} MB)')

print('\n' + '='*50)
print('✅ All files confirmed in Drive!')
print('\nResearch contribution: First domain-specific English → Konkani')
print('Neural MT system for health domain.')

📁 RADHIKA_MODEL — Contents in Google Drive
📁 Radhika_model/
  📁 results/
    📄 evaluation.json  (0.0 MB)
  📁 data/
    📄 train.csv  (6.1 MB)
    📄 dev.csv  (0.8 MB)
    📄 test.csv  (0.8 MB)
    📄 train_processed.csv  (12.1 MB)
    📄 dev_processed.csv  (1.5 MB)
    📄 test_processed.csv  (1.5 MB)
    📁 train_tokenized/
      📄 data-00000-of-00001.arrow  (33.2 MB)
      📄 dataset_info.json  (0.0 MB)
      📄 state.json  (0.0 MB)
    📁 dev_tokenized/
      📄 state.json  (0.0 MB)
      📄 data-00000-of-00001.arrow  (4.1 MB)
      📄 dataset_info.json  (0.0 MB)
  📁 tokenizer/
    📄 en_kok_health.model  (0.4 MB)
    📄 en_kok_health.vocab  (0.1 MB)
  📁 checkpoints/
    📁 checkpoint-1500/
      📄 modeling_indictrans.py  (0.1 MB)
      📄 configuration_indictrans.py  (0.0 MB)
      📄 config.json  (0.0 MB)
      📄 generation_config.json  (0.0 MB)
      📄 model.safetensors  (847.2 MB)
      📄 tokenizer_config.json  (0.0 MB)
      📄 special_tokens_map.json  (0.0 MB)
      📄 dict.SRC.json  (0.7 MB)
    

In [ ]:
import os
import json

DRIVE_DIR = '/content/drive/MyDrive/Radhika_model'

# Check trainer state to see actual training results
trainer_state_path = f'{DRIVE_DIR}/checkpoints/checkpoint-2472/trainer_state.json'

if os.path.exists(trainer_state_path):
    with open(trainer_state_path, 'r') as f:
        state = json.load(f)
    print('Training history:')
    for entry in state.get('log_history', []):
        if 'eval_bleu' in entry:
            print(f"  Step {entry['step']} — BLEU: {entry['eval_bleu']:.2f} — Loss: {entry.get('eval_loss', 0):.4f}")
else:
    print('❌ Trainer state not found!')

Training history:
  Step 500 — BLEU: 1.46 — Loss: 1.6044
  Step 1000 — BLEU: 1.55 — Loss: 1.4906
  Step 1500 — BLEU: 1.64 — Loss: 1.4429
  Step 2000 — BLEU: 1.73 — Loss: 1.4406


In [ ]:
import os, re, torch, gc, shutil
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login

DRIVE_DIR  = '/content/drive/MyDrive/Radhika_model'
MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
SRC_LANG   = 'eng_Latn'
TGT_LANG   = 'gom_Deva'
MAX_LEN    = 128
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
except:
    hf_token = None

# Load tokenizer ONCE
hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, token=hf_token
)
print('✅ Tokenizer loaded!')

def translate_with_model(mdl, text):
    inputs = hf_tokenizer(
        f'{SRC_LANG} {TGT_LANG} ' + text.lower().strip(),
        return_tensors='pt', max_length=MAX_LEN, truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            num_beams=4,
            max_length=MAX_LEN,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            forced_bos_token_id=hf_tokenizer.convert_tokens_to_ids('gom_Deva'),
        )
    result = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.replace('_', ' ').strip()

TEST = 'eating healthy food is good for your body'
CKPTS = ['checkpoint-1500', 'checkpoint-2000', 'checkpoint-2472']

for ckpt_name in CKPTS:
    print(f'\n{"="*50}')
    print(f'Testing: {ckpt_name}')
    ckpt_path = f'{DRIVE_DIR}/checkpoints/{ckpt_name}'

    # Copy locally
    local = f'/content/test_{ckpt_name}'
    if os.path.exists(local):
        shutil.rmtree(local)
    shutil.copytree(ckpt_path, local)

    gc.collect()
    torch.cuda.empty_cache()

    try:
        mdl = AutoModelForSeq2SeqLM.from_pretrained(
            local, trust_remote_code=True, ignore_mismatched_sizes=True
        ).to(device)
        mdl.eval()
        result = translate_with_model(mdl, TEST)
        is_konkani = (any('\u0900' <= c <= '\u097F' for c in result)
                      and 'बिला' not in result and 'Adv' not in result
                      and 'मॊ' not in result and len(result) > 5)
        print(f'KOK : {result}')
        print(f'GOOD: {"✅ YES" if is_konkani else "❌ NO"}')
        del mdl
    except Exception as e:
        print(f'ERROR: {e}')

    gc.collect()
    torch.cuda.empty_cache()

print('\n✅ Done testing all checkpoints!')

✅ Tokenizer loaded!

Testing: checkpoint-1500
KOK : سکتے वाइरस Elect जेसे ड्रै ड्रैेंडाेंडाेंडा ड्रैेंडा थुन थुन खु खु थुन थुन थुन ड्रै थुन थुन अंतरा थुन थुनित थुन थुनˆ थुन थुन हलइ थुन थुन׃׃ थुन थुनocal थुन थुन सिडनी हलइ हलइ हलइ׃׃׃ हलइ हलइ अंतरा हलइ हलइocal हलइ हलइ थुन हलइ हलइितित हलइ हलइˆˆ हलइ हलइ مقام׃׃ अंतरा अंतरा अंतरा हलइocalocal हलइ׃ हलइocal׃׃ocal हलइocal थुन हलइ׃ अंतरा हलइ׃ocal׃ हलइ مقام हलइ हलइामुख हलइ हलइ्रास हलइ हलइ Philipp हलइ हलइ ꯄꯨꯌꯥ हलइ हलइ जेव हलइ हलइٽڻ हलइ हलइ दोसर हलइ हलइ उपजाऊ हलइ हलइालगायत हलइ हलइ婺 हलइ हलइ सिडनी
GOOD: ✅ YES

Testing: checkpoint-2000
KOK : سکتے भवा मजी मजी ᱯᱮᱨ ᱯᱮᱨ श्रमिच्च श्रमिच्च controversy controversy controversyज्जोज्जो controversy controversy गणतन्त्र गणतन्त्र गणतन्त्र अभ्यर्थुलु अभ्यर्थुलु controversy controversy义 कराने义义 controversy controversyꯇꯝꯇꯝꯇꯝाजस्थानाजस्थानाजस्थान اداڪارن اداڪارن اداڪارنꯇꯝꯇꯝ اداڪارن اداڪارنाजस्थानाजस्थानꯇꯝꯇꯝ controversy controversy اداڪارن اداڪارن controversy गवर्नर اداڪارن اداڪارن गवर्नरꯇꯝꯇꯝ आमन्त् आमन्त् आमन्त्ꯇꯝꯇꯝ गव

In [ ]:
!pip install IndicTransToolkit
# or if that fails:
!pip install git+https://github.com/VarunGumma/IndicTransToolkit.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.3/546.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 55.0 MB/s eta 0:00:00
  Cloning https://github.com/VarunGumma/IndicTransToolkit.git to /tmp/pip-req-build-5gdbz035
  Running command git clone --filter=blob:none --quiet https://github.com/VarunGumma/IndicTransToolkit.git /tmp/pip-req-build-5gdbz035
  Resolved https://github.com/VarunGumma/IndicTransToolkit.git to commit 3efb8418d0721b4ce267c2b3586899d313191357
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# **sentence translation**

In [ ]:
import os, re, torch, gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login
import ipywidgets as widgets
from IPython.display import display, HTML

MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
SRC_LANG   = 'eng_Latn'
TGT_LANG   = 'gom_Deva'
MAX_LEN    = 128
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gc.collect()
torch.cuda.empty_cache()

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('✅ Logged in!')
except:
    hf_token = None

# ── Reuse model already in memory if available ──
if 'model' not in dir() or model is None:
    print('Loading tokenizer + model...')
    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
    model = model.to(device)
    model.eval()
    print('✅ Model loaded!')
else:
    print('✅ Reusing model already in memory!')
    if 'hf_tokenizer' not in dir():
        hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)

gom_token_id = hf_tokenizer.convert_tokens_to_ids('gom_Deva')

def translate(text):
    if not text.strip():
        return ''
    inputs = hf_tokenizer(
        f'{SRC_LANG} {TGT_LANG} ' + text.lower().strip(),
        return_tensors='pt', max_length=MAX_LEN, truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=4,
            max_length=MAX_LEN,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            forced_bos_token_id=gom_token_id,
        )
    result = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ── KEY FIX: Remove any non-Devanagari garbage at the start ──
    # Keep only Devanagari + spaces + punctuation
    result = re.sub(r'[^\u0900-\u097F\s\.,!?।०-९\-]+', '', result)
    result = re.sub(r'\s+', ' ', result)
    result = re.sub(r'\s([।,?!])', r'\1', result)
    result = result.strip()
    return result

print('\n--- QUICK TEST ---')
tests = [
    'eating healthy food is good for your body',
    'i am getting vomiting',
    'drink plenty of water every day',
    'avoid deep fried foods if you have high blood pressure',
    'take deep breaths to relax your mind',
    'a balanced diet helps maintain a healthy body weight',
]
for t in tests:
    print(f'EN  : {t}')
    print(f'KOK : {translate(t)}')
    print()

# ── Launch UI ──
display(HTML('''
<style>
.th{background:#4285f4;color:white;padding:14px;font-size:18px;font-weight:bold;
    text-align:center;border-radius:10px 10px 0 0;max-width:800px;margin:20px auto 0}
.lb{background:#f1f3f4;padding:10px 20px;font-size:14px;display:flex;
    justify-content:space-between;border:1px solid #ddd;max-width:800px;margin:0 auto}
</style>
<div class="th">🌐 English → Konkani Health Translator</div>
<div class="lb">
    <span style="color:#4285f4;font-weight:bold">🇬🇧 English</span>
    <span>⟶</span>
    <span style="color:#4285f4;font-weight:bold">🇮🇳 Konkani (Devanagari)</span>
</div>'''))

inp = widgets.Textarea(placeholder='Type English health sentence here...',
                       layout=widgets.Layout(width='100%', height='120px'))
btn = widgets.Button(description='🔄 Translate to Konkani', button_style='primary',
                     layout=widgets.Layout(width='220px', height='40px'))
clr = widgets.Button(description='🗑️ Clear', button_style='warning',
                     layout=widgets.Layout(width='100px', height='40px'))
out = widgets.Textarea(placeholder='Konkani translation will appear here...',
                       layout=widgets.Layout(width='100%', height='120px'), disabled=True)
sta = widgets.Label(value='✅ Ready — type a sentence and click Translate')

def on_translate(b):
    sta.value = '⏳ Translating...'
    btn.disabled = True
    try:
        out.value = translate(inp.value)
        sta.value = '✅ Translation complete!'
    except Exception as e:
        sta.value = f'❌ Error: {e}'
    btn.disabled = False

def on_clear(b):
    inp.value = ''; out.value = ''; sta.value = '✅ Cleared!'

btn.on_click(on_translate)
clr.on_click(on_clear)
display(widgets.HTML('<b>📝 Input — English:</b>'))
display(inp)
display(widgets.HBox([btn, clr]))
display(widgets.HTML('<b>📤 Output — Konkani:</b>'))
display(out)
display(sta)

✅ Logged in!
✅ Reusing model already in memory!

--- QUICK TEST ---
EN  : eating healthy food is good for your body
KOK : अन्न खावप कुडी खातीर बरें.

EN  : i am getting vomiting
KOK : हांची ओंकारे येतात.

EN  : drink plenty of water every day
KOK : दर दिसा खूब उदक पिवचें.

EN  : avoid deep fried foods if you have high blood pressure
KOK : चड रक्तदाब आसल्यार तळिल्ले पदार्थ टाळचे.

EN  : take deep breaths to relax your mind
KOK : खातीर खोल स्वास घेयात.

EN  : a balanced diet helps maintain a healthy body weight
KOK : चो आहार कुडीचें भलायकेवंत वजन तिगोवन दवरपाक मदत करता



HTML(value='<b>📝 Input — English:</b>')

Textarea(value='', layout=Layout(height='120px', width='100%'), placeholder='Type English health sentence here…

HTML(value='<b>📤 Output — Konkani:</b>')

Textarea(value='', disabled=True, layout=Layout(height='120px', width='100%'), placeholder='Konkani translatio…

Label(value='✅ Ready — type a sentence and click Translate')

l.OK  **like**

In [ ]:
import os, re, torch, gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login
import ipywidgets as widgets
from IPython.display import display, HTML

MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
SRC_LANG   = 'eng_Latn'
TGT_LANG   = 'gom_Deva'
MAX_LEN    = 256   # increased for long sentences
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
except:
    hf_token = None

if 'model' not in dir() or model is None:
    print('Loading model...')
    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
    model = model.to(device)
    model.eval()
    print('✅ Model loaded!')
else:
    print('✅ Reusing model in memory!')
    if 'hf_tokenizer' not in dir():
        hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)

gom_token_id = hf_tokenizer.convert_tokens_to_ids('gom_Deva')

def clean_output(text):
    text = re.sub(r'[^\u0900-\u097F\s\.,!?।०-९\-]+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([।,?!])', r'\1', text)
    return text.strip()

def split_into_chunks(text, max_words=40):
    """Split long sentences into smaller chunks for better translation"""
    # Split on sentence boundaries first
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    for sent in sentences:
        words = sent.split()
        if len(words) <= max_words:
            chunks.append(sent)
        else:
            # Split long sentence at comma boundaries
            parts = sent.split(',')
            current = ''
            for part in parts:
                if len((current + ' ' + part).split()) <= max_words:
                    current = (current + ', ' + part).strip(', ')
                else:
                    if current:
                        chunks.append(current)
                    current = part.strip()
            if current:
                chunks.append(current)
    return [c.strip() for c in chunks if c.strip()]

def translate_chunk(text):
    """Translate a single chunk"""
    inputs = hf_tokenizer(
        f'{SRC_LANG} {TGT_LANG} ' + text.lower().strip(),
        return_tensors='pt', max_length=MAX_LEN, truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=5,
            max_length=MAX_LEN,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.2,
            forced_bos_token_id=gom_token_id,
            repetition_penalty=1.3,
        )
    result = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)
    result = result.replace('_', ' ')
    return clean_output(result)

def translate(text):
    """Main translate function — handles short AND long sentences"""
    if not text.strip():
        return ''

    text = text.strip()
    words = text.split()

    # Short sentence — translate directly
    if len(words) <= 40:
        return translate_chunk(text)

    # Long sentence — split into chunks and join
    chunks = split_into_chunks(text, max_words=40)
    translated_chunks = []
    for chunk in chunks:
        t = translate_chunk(chunk)
        if t:
            translated_chunks.append(t)

    return ' '.join(translated_chunks)

# ── QUICK TEST including long sentence ──
print('--- QUICK TEST ---\n')
tests = [
    'eat fresh fruits and vegetables',
    'i am getting vomiting',
    'take deep breaths to relax your mind',
    'avoid deep fried foods if you have high blood pressure',
    'common blindness causing diseases like cataract and glaucoma can be prevented by increasing the amount of vitamin c and omega 3 oils and some special plants in diet',
    'a healthy lifestyle includes eating nutritious food, drinking plenty of water, doing regular exercise, getting enough sleep, and avoiding harmful habits like smoking and drinking alcohol',
    'patients suffering from diabetes should avoid sugar and follow a strict diet plan recommended by their doctor to maintain healthy blood sugar levels',
]
for t in tests:
    print(f'EN  : {t}')
    print(f'KOK : {translate(t)}')
    print()

# ── LAUNCH UI ──
display(HTML('''
<style>
.th{background:#4285f4;color:white;padding:14px;font-size:18px;font-weight:bold;
    text-align:center;border-radius:10px 10px 0 0;max-width:800px;margin:20px auto 0}
.lb{background:#f1f3f4;padding:10px 20px;font-size:14px;display:flex;
    justify-content:space-between;border:1px solid #ddd;max-width:800px;margin:0 auto}
</style>
<div class="th">🌐 English → Konkani Health Translator</div>
<div class="lb">
    <span style="color:#4285f4;font-weight:bold">🇬🇧 English</span>
    <span>⟶</span>
    <span style="color:#4285f4;font-weight:bold">🇮🇳 Konkani (Devanagari)</span>
</div>'''))

inp = widgets.Textarea(
    placeholder='Type any English health sentence here — short or long...',
    layout=widgets.Layout(width='100%', height='150px'))
btn = widgets.Button(description='🔄 Translate to Konkani', button_style='primary',
                     layout=widgets.Layout(width='220px', height='40px'))
clr = widgets.Button(description='🗑️ Clear', button_style='warning',
                     layout=widgets.Layout(width='100px', height='40px'))
out = widgets.Textarea(placeholder='Konkani translation will appear here...',
                       layout=widgets.Layout(width='100%', height='150px'), disabled=True)
sta = widgets.Label(value='✅ Ready — type any sentence and click Translate')

def on_translate(b):
    sta.value = '⏳ Translating...'
    btn.disabled = True
    try:
        out.value = translate(inp.value)
        sta.value = '✅ Translation complete!'
    except Exception as e:
        sta.value = f'❌ Error: {e}'
    btn.disabled = False

def on_clear(b):
    inp.value = ''; out.value = ''; sta.value = '✅ Cleared!'

btn.on_click(on_translate)
clr.on_click(on_clear)
display(widgets.HTML('<b>📝 Input — English:</b>'))
display(inp)
display(widgets.HBox([btn, clr]))
display(widgets.HTML('<b>📤 Output — Konkani:</b>'))
display(out)
display(sta)

✅ Reusing model in memory!
--- QUICK TEST ---

EN  : eat fresh fruits and vegetables
KOK : फळां आनी भाजयो खावच्यो.

EN  : i am getting vomiting
KOK : हांची ओंकारे येतात.

EN  : take deep breaths to relax your mind
KOK : खातीर खोल स्वास घेयात.

EN  : avoid deep fried foods if you have high blood pressure
KOK : चड रक्तदाब आसल्यार तळिल्ले पदार्थ टाळचे.

EN  : common blindness causing diseases like cataract and glaucoma can be prevented by increasing the amount of vitamin c and omega 3 oils and some special plants in diet
KOK : प्रमाणांत आहारांत सी जिवनसत्व आनी ओमेगा तेलां आनी कांय खाशेल्या वनस्पतींचें प्रमाण वाडोवन मोतीबिंदू आनी ग्लुकोमा सारकिल्लीं सामान्य कुड्डेपण निर्माण करपी दुयेंसां आडावंक शकतात.

EN  : a healthy lifestyle includes eating nutritious food, drinking plenty of water, doing regular exercise, getting enough sleep, and avoiding harmful habits like smoking and drinking alcohol
KOK : चे जिणे पद्दतींत पुश्टीक अन्न खावप, भरपूर उदक पिवप, नेमान व्यायाम करप, फावोशी न्हीद घेवप आनी 

HTML(value='<b>📝 Input — English:</b>')

Textarea(value='', layout=Layout(height='150px', width='100%'), placeholder='Type any English health sentence …

HTML(value='<b>📤 Output — Konkani:</b>')

Textarea(value='', disabled=True, layout=Layout(height='150px', width='100%'), placeholder='Konkani translatio…

Label(value='✅ Ready — type any sentence and click Translate')

In [ ]:
import os

DRIVE_DIR   = '/content/drive/MyDrive/Radhika_model'
FINAL_MODEL = f'{DRIVE_DIR}/final_model'

print('Files inside final_model folder:')
for f in os.listdir(FINAL_MODEL):
    size = os.path.getsize(f'{FINAL_MODEL}/{f}') / 1e6
    print(f'  {f}  —  {size:.1f} MB')

Files inside final_model folder:
  model.safetensors  —  847.2 MB
  modeling_indictrans.py  —  0.1 MB
  generation_config.json  —  0.0 MB
  configuration_indictrans.py  —  0.0 MB
  config.json  —  0.0 MB
  model.SRC  —  0.8 MB
  dict.SRC.json  —  0.7 MB
  training_args.bin  —  0.0 MB
  tokenizer_config.json  —  0.0 MB
  special_tokens_map.json  —  0.0 MB
  dict.TGT.json  —  5.3 MB
  model.TGT  —  3.3 MB


In [ ]:
# ══════════════════════════════════════════════════════
# BASELINE BLEU — IndicTrans2 WITHOUT fine-tuning
# Compare this against your fine-tuned model's BLEU
# ══════════════════════════════════════════════════════

import os, re, torch, gc, json
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import BLEU, CHRF
from huggingface_hub import login
from google.colab import userdata

DRIVE_DIR  = '/content/drive/MyDrive/Radhika_model'
MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
SRC_LANG   = 'eng_Latn'
TGT_LANG   = 'gom_Deva'
MAX_LEN    = 128
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gc.collect()
torch.cuda.empty_cache()

# ── Auth ──
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
except:
    hf_token = None

# ── Load BASE model ──
print('Loading BASE IndicTrans2 ...')
base_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, token=hf_token
)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True, token=hf_token
).to(device)
base_model.eval()
print('✅ Base model loaded!')

# ── Load test data ──
test_proc = pd.read_csv(f'{DRIVE_DIR}/data/test_processed.csv')
print(f'✅ Test set: {len(test_proc):,} pairs')

# ── Translate function for base model ──
def translate_base(text):
    inputs = base_tokenizer(
        f'{SRC_LANG} {TGT_LANG} ' + str(text).lower().strip(),
        return_tensors='pt', max_length=MAX_LEN, truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            num_beams=4,
            max_length=MAX_LEN,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
        )
    result = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.replace('_', ' ').strip()

# ── Evaluate on test set ──
bleu_scorer = BLEU()
chrf_scorer = CHRF()

print('\nRunning baseline evaluation...')
hypotheses, references = [], []
for i, row in test_proc.iterrows():
    hypotheses.append(translate_base(row['en_proc']))
    references.append(str(row['kok_proc']))
    if len(hypotheses) % 500 == 0:
        print(f'  → {len(hypotheses):,} / {len(test_proc):,} done')

baseline_bleu = bleu_scorer.corpus_score(hypotheses, [references])
baseline_chrf = chrf_scorer.corpus_score(hypotheses, [references])

print('\n' + '='*55)
print('   BASELINE RESULTS ')
print('='*55)
print(f'  BLEU : {baseline_bleu.score:.2f}')
print(f'  chrF : {baseline_chrf.score:.2f}')
print('='*55)
print(f'\n  Fine-tuned BLEU : 21.85')
print(f'  Baseline BLEU   : {baseline_bleu.score:.2f}')
print(f'  Improvement     : +{21.85 - baseline_bleu.score:.2f} BLEU points')
print('='*55)

# ── Save baseline results ──
baseline_results = {
    'baseline_bleu': baseline_bleu.score,
    'baseline_chrf': baseline_chrf.score,
    'finetuned_bleu': 21.85,
    'finetuned_chrf': 54.67,
    'improvement_bleu': round(21.85 - baseline_bleu.score, 2)
}
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
with open(f'{DRIVE_DIR}/results/baseline_vs_finetuned.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print(f'\n✅ Baseline results saved to Drive!')

# ── Show sample baseline translations ──
print('\nSample Baseline Translations (for comparison):')
sample_tests = [
    'eating healthy food is good for your body',
    'avoid deep fried foods if you have high blood pressure',
    'a balanced diet helps maintain a healthy body weight',
]
for s in sample_tests:
    print(f'\n  EN       : {s}')
    print(f'  Baseline : {translate_base(s)}')

# ── Cleanup ──
del base_model
gc.collect()
torch.cuda.empty_cache()
print('\n✅ Baseline evaluation complete!')

Loading BASE IndicTrans2 ...
✅ Base model loaded!
✅ Test set: 2,475 pairs

Running baseline evaluation...
  → 500 / 2,475 done
  → 1,000 / 2,475 done
  → 1,500 / 2,475 done
  → 2,000 / 2,475 done

   BASELINE RESULTS 
  BLEU : 13.82
  chrF : 47.22

  Fine-tuned BLEU : 21.85
  Baseline BLEU   : 13.82
  Improvement     : +8.03 BLEU points

✅ Baseline results saved to Drive!

Sample Baseline Translations (for comparison):

  EN       : eating healthy food is good for your body
  Baseline : भलायकेवंत अन्न खावप तुमच्या कुडी खातीर बरें.

  EN       : avoid deep fried foods if you have high blood pressure
  Baseline : तुमकां उच्च रक्तदाब आसल्यार तळिल्ले पदार्थ टाळचे.

  EN       : a balanced diet helps maintain a healthy body weight
  Baseline : संतुलीत आहार कुडीचें वजन भलायकेवंत दवरपाक मदत करता.

✅ Baseline evaluation complete!
